# **🏋🏽 Wearable Biomechanics - Dataset Manager**


### 🔧 Colab Setup & Library Init

In [ ]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & set Plotly renderer
try:
    from google.colab import drive
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("open_dataset_store") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/janithcyapa/OpenDatasetStore.git"])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "cleannjerk_dataset" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import init_store, display_table

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}
store = init_store(dataset_path=dataset_path)

# **👤 People Management (Subjects)**


### 📝 Register New Subject

In [ ]:
# @title 📝 Register New Subject { display-mode: "form" }
name = "Dinithi" # @param {type:"string"}
age = 23 # @param {type:"number"}
weight_kg = 0 # @param {type:"number"}
height_m = 0 # @param {type:"number"}
achievement = "-" # @param {type:"string"}
best_weight = "-" # @param {type:"string"}
experience_years = 1 # @param {type:"integer"}
notes = "-" # @param {type:"string"}

sub_id = store.create_entity(
    entity_type="subjects",
    name=name,
    age=age,
    weight=weight_kg,
    height=height_m,
    experience=experience_years,
    achievement=achievement,
    best_weight=best_weight,
    notes=notes,
)
print(f"✅ Created Subject ID: {sub_id}")

### 📋 Participant Registry

In [ ]:
# @title 📋 Participant Registry { display-mode: "form" }
df_subjects = store.list_entities("subjects")
print(f"Total Subjects Registered: {len(df_subjects)}")
display_table(df_subjects)

# **📂 Experiment Upload & Data Exploration**


### 🚀 Upload New Experiment

In [ ]:
# @title 🚀 Upload New Experiment { display-mode: "form" }
from src.imu import plot_quick_accel_check

target_subject_id = "sub_0019" # @param {type:"string"}
activity_type = "Clean & Jerk" # @param {type:"string"}
weight_used = 30 # @param {type:"number"}
date = "2026-11-03" # @param {type:"string"}
time = "12:27" # @param {type:"string"}
timestamp = 1793689041 # @param {type:"number"}
notes = "3 reps, Complete" # @param {type:"string"}

exp_id = store.create_entry_interactive(
    entry_type="experiments",
    entity_id=target_subject_id,
    date=date,
    time=time,
    description=activity_type,
    timestamp=timestamp,
    weight_kg=weight_used,
    notes=notes,
)

if exp_id:
    raw_df = store.get_entry_raw_data("experiments", exp_id)
    plot_quick_accel_check(raw_df, exp_id).show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})

### 📋 Experiments Browser

In [ ]:
# @title 📋 Experiments Browser { display-mode: "form" }
include_paths = True # @param {type:"boolean"}
df_experiments = store.list_entries("experiments", include_paths=include_paths)
if not df_experiments.empty:
    df_experiments = df_experiments.sort_values(by="entry_id", ascending=False)
print(f"Total Experiments: {len(df_experiments)}")
display_table(df_experiments)

### 📎 Attachment Manager

In [ ]:
# @title 📎 Attachment Manager { display-mode: "form" }
experiment_to_attach = "experiments_0060" # @param {type:"string"}
file_label = "side_video" # @param ["front_video", "side_video", "raw_zip", "other"]

rel_path = store.upload_related_file_interactive(
    entry_type="experiments",
    entry_id=experiment_to_attach,
    label=file_label,
)

# **📊 Advanced IMU Analysis**


### 🔍 Detailed Signal Analysis

In [ ]:
# @title 🔍 Detailed Signal Analysis { display-mode: "form" }
from src.imu import plot_health_matrix, plot_imu_verification_dashboard

selected_exp = "experiments_0055" # @param {type:"string"}
df_analysis = store.get_entry_raw_data("experiments", selected_exp)

health_fig = plot_health_matrix(df_analysis, title=f"{selected_exp} Sensor Health Matrix")
if health_fig:
    health_fig.show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})
plot_imu_verification_dashboard(df_analysis, time_col="Timestamp").show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})